# LSTM-based PM2.5 Prediction Model

This notebook implements an LSTM model to predict PM2.5 levels using AirNow/AQS data.
- Input: 5 time frames
- Output: 5 time frames (future predictions)
- Includes proper forward/back filling with trust values
- Adds temporal features for better predictions

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path
import sqlite3
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import warnings
warnings.filterwarnings('ignore')

print("TensorFlow version:", tf.__version__)
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

## 1. Data Loading and Preprocessing

In [ ]:
# Connect to database and load data
db_path = Path('../data/aqs_data.db')

if not db_path.exists():
    print(f"Error: Database not found at {db_path}")
    print("Please ensure the AQS data has been loaded into the database.")
else:
    conn = sqlite3.connect(db_path)
    
    # Load California data for Site 0010 (you can change this to 0019 or other sites)
    query = """
    SELECT "Date GMT", "Time GMT", "Sample Measurement", "Site Num"
    FROM hourly_88101_2020
    WHERE "State Name" = 'California' AND "Site Num" = '0010'
    ORDER BY "Date GMT", "Time GMT"
    """
    
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    print(f"Loaded {len(df)} records")
    print(f"Date range: {df['Date GMT'].min()} to {df['Date GMT'].max()}")
    print(f"\nFirst few rows:")
    print(df.head())

In [ ]:
# Create datetime column
df['datetime'] = pd.to_datetime(df['Date GMT'] + ' ' + df['Time GMT'])
df = df.sort_values('datetime').reset_index(drop=True)

# Convert Sample Measurement to numeric
df['Sample Measurement'] = pd.to_numeric(df['Sample Measurement'], errors='coerce')

print(f"\nData after datetime creation:")
print(f"Shape: {df.shape}")
print(f"Datetime range: {df['datetime'].min()} to {df['datetime'].max()}")
print(f"Missing values in Sample Measurement: {df['Sample Measurement'].isna().sum()}")

## 2. Create Complete Hourly Time Series

We need to create a complete hourly time series to identify missing data points.

In [ ]:
# Create a complete hourly datetime range
start_date = df['datetime'].min()
end_date = df['datetime'].max()
complete_datetime_range = pd.date_range(start=start_date, end=end_date, freq='h')

print(f"Original data points: {len(df)}")
print(f"Complete hourly range: {len(complete_datetime_range)}")
print(f"Missing data points: {len(complete_datetime_range) - len(df)}")

# Create a complete dataframe with all hourly timestamps
df_complete = pd.DataFrame({'datetime': complete_datetime_range})
df_complete = df_complete.merge(df[['datetime', 'Sample Measurement']], on='datetime', how='left')

print(f"\nComplete dataframe shape: {df_complete.shape}")
print(f"Missing values: {df_complete['Sample Measurement'].isna().sum()}")

## 3. Trust Value Calculation Functions

Implement trust values based on:
- Original observations get trust = 1.0
- Forward filled values get trust based on decay
- Backward filled values get trust based on decay

In [ ]:
# Import shared utility functions
import sys
sys.path.append("../scripts")
from lstm_utils import calculate_trust_values, add_time_features, create_sequences, denormalize_pm25

print("Trust calculation function imported.")

In [ ]:
# Apply trust calculation
df_complete = calculate_trust_values(df_complete, 'Sample Measurement', decay_rate=0.9)

print("Trust values calculated.")
print(f"\nTrust value statistics:")
print(df_complete['trust'].describe())
print(f"\nFill method distribution:")
print(df_complete['fill_method'].value_counts())

In [ ]:
# Now perform forward and backward filling
df_complete['Sample Measurement'] = df_complete['Sample Measurement'].ffill().bfill()

print(f"After filling - Missing values: {df_complete['Sample Measurement'].isna().sum()}")
print(f"\nData summary:")
print(df_complete[['Sample Measurement', 'trust', 'fill_method']].describe())

## 4. Add Time Features

Extract temporal features that can help the LSTM model understand patterns.

In [ ]:
# Time features are imported from lstm_utils
# add_time_features function adds temporal features to the dataframe

# Add time features
df_complete = add_time_features(df_complete)

print("Time features added:")
print(df_complete.columns.tolist())
print(f"
Dataframe shape: {df_complete.shape}")
print(df_complete.head())

## 5. Visualization with Trust-based Color Coding

In [ ]:
# Create visualization with color coding based on trust values
# Sample the data for better visualization (every 24th point = daily)
df_plot = df_complete.iloc[::24].copy()

fig = go.Figure()

# Add scatter plot with color based on trust
fig.add_trace(go.Scatter(
    x=df_plot['datetime'],
    y=df_plot['Sample Measurement'],
    mode='markers+lines',
    marker=dict(
        size=6,
        color=df_plot['trust'],
        colorscale='RdYlGn',  # Red (low trust) to Green (high trust)
        showscale=True,
        colorbar=dict(title="Trust Value"),
        cmin=0,
        cmax=1
    ),
    line=dict(width=0.5, color='lightgray'),
    text=[f"Trust: {t:.2f}<br>Method: {m}" for t, m in zip(df_plot['trust'], df_plot['fill_method'])],
    hovertemplate='%{text}<br>PM2.5: %{y:.2f}<br>Time: %{x}<extra></extra>',
    name='PM2.5'
))

fig.update_layout(
    title='PM2.5 Measurements with Trust-based Color Coding (Daily Samples)',
    xaxis_title='Date',
    yaxis_title='PM2.5 (µg/m³)',
    hovermode='closest',
    height=600
)

fig.show()

In [ ]:
# Another visualization showing fill methods
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Sample every 12 hours for clearer visualization
df_plot2 = df_complete.iloc[::12].copy()

# Plot 1: PM2.5 with color by fill method
colors = {'original': 'green', 'forward': 'orange', 'backward': 'blue', 'missing': 'red'}
for method, color in colors.items():
    mask = df_plot2['fill_method'] == method
    if mask.any():
        axes[0].scatter(df_plot2.loc[mask, 'datetime'], 
                       df_plot2.loc[mask, 'Sample Measurement'],
                       c=color, label=method, alpha=0.6, s=20)

axes[0].plot(df_plot2['datetime'], df_plot2['Sample Measurement'], 
             color='lightgray', alpha=0.3, linewidth=0.5)
axes[0].set_xlabel('Date')
axes[0].set_ylabel('PM2.5 (µg/m³)')
axes[0].set_title('PM2.5 Measurements by Fill Method')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Trust values over time
axes[1].scatter(df_plot2['datetime'], df_plot2['trust'], 
               c=df_plot2['trust'], cmap='RdYlGn', s=20)
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Trust Value')
axes[1].set_title('Trust Values Over Time')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Prepare Data for LSTM Model

Create sequences: 5 frames in, 5 frames out

In [ ]:
# Select features for the model
feature_cols = ['Sample Measurement', 'trust', 'hour_sin', 'hour_cos', 
                'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_weekend']

# Prepare feature matrix
features = df_complete[feature_cols].values

# Normalize features
scaler = MinMaxScaler()
features_scaled = scaler.fit_transform(features)

print(f"Feature matrix shape: {features_scaled.shape}")
print(f"Features: {feature_cols}")

In [ ]:
# create_sequences is imported from lstm_utils
# It creates sequences for LSTM: n_steps_in -> n_steps_out

# Create sequences
n_steps_in = 5
n_steps_out = 5

X, y = create_sequences(features_scaled, n_steps_in, n_steps_out)

print(f"Input sequences shape (X): {X.shape}")
print(f"Output sequences shape (y): {y.shape}")
print(f"
X shape interpretation: (samples, time_steps_in, features)")
print(f"y shape interpretation: (samples, time_steps_out)")

In [ ]:
# Split data into train, validation, and test sets
# Use 70% for training, 15% for validation, 15% for testing
train_size = int(0.7 * len(X))
val_size = int(0.15 * len(X))

X_train = X[:train_size]
y_train = y[:train_size]

X_val = X[train_size:train_size + val_size]
y_val = y[train_size:train_size + val_size]

X_test = X[train_size + val_size:]
y_test = y[train_size + val_size:]

print(f"Training set: X={X_train.shape}, y={y_train.shape}")
print(f"Validation set: X={X_val.shape}, y={y_val.shape}")
print(f"Test set: X={X_test.shape}, y={y_test.shape}")

## 7. Build LSTM Model

In [ ]:
def build_lstm_model(n_steps_in, n_features, n_steps_out):
    """
    Build LSTM model for time series prediction.
    
    Parameters:
    -----------
    n_steps_in : number of input time steps
    n_features : number of features per time step
    n_steps_out : number of output time steps
    
    Returns:
    --------
    Compiled Keras model
    """
    model = Sequential([
        # First LSTM layer with return sequences
        LSTM(64, activation='tanh', return_sequences=True, 
             input_shape=(n_steps_in, n_features)),
        Dropout(0.2),
        
        # Second LSTM layer
        LSTM(32, activation='tanh', return_sequences=False),
        Dropout(0.2),
        
        # Dense layers
        Dense(32, activation='relu'),
        Dropout(0.2),
        
        # Output layer
        Dense(n_steps_out)
    ])
    
    model.compile(
        optimizer='adam',
        loss='mse',
        metrics=['mae']
    )
    
    return model

# Build the model
n_features = X_train.shape[2]
model = build_lstm_model(n_steps_in, n_features, n_steps_out)

print("Model architecture:")
model.summary()

## 8. Train the Model

In [ ]:
# Define callbacks
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

# Train the model
print("Training the model...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

print("\nTraining complete!")

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss plot
axes[0].plot(history.history['loss'], label='Training Loss')
axes[0].plot(history.history['val_loss'], label='Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (MSE)')
axes[0].set_title('Model Loss During Training')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE plot
axes[1].plot(history.history['mae'], label='Training MAE')
axes[1].plot(history.history['val_mae'], label='Validation MAE')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].set_title('Model MAE During Training')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Evaluate the Model

In [ ]:
# Evaluate on test set
test_loss, test_mae = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss (MSE): {test_loss:.6f}")
print(f"Test MAE: {test_mae:.6f}")

# Make predictions
y_pred = model.predict(X_test, verbose=0)

print(f"\nPredictions shape: {y_pred.shape}")
print(f"Expected shape: {y_test.shape}")

In [ ]:
# denormalize_pm25 is imported from lstm_utils
# It handles denormalization of PM2.5 values back to original scale

# Denormalize predictions and actual values for each time step
y_test_denorm = np.zeros_like(y_test)
y_pred_denorm = np.zeros_like(y_pred)

for t in range(n_steps_out):
    y_test_denorm[:, t] = denormalize_pm25(y_test[:, t], scaler, n_features)
    y_pred_denorm[:, t] = denormalize_pm25(y_pred[:, t], scaler, n_features)

print(f"Denormalized predictions shape: {y_pred_denorm.shape}")
print(f"Sample actual values (first sequence, all 5 time steps): {y_test_denorm[0]}")
print(f"Sample predictions (first sequence, all 5 time steps): {y_pred_denorm[0]}")

In [ ]:
# Calculate metrics for each prediction horizon
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Performance metrics for each prediction horizon:")
print("="*60)

for t in range(n_steps_out):
    mae = mean_absolute_error(y_test_denorm[:, t], y_pred_denorm[:, t])
    rmse = np.sqrt(mean_squared_error(y_test_denorm[:, t], y_pred_denorm[:, t]))
    r2 = r2_score(y_test_denorm[:, t], y_pred_denorm[:, t])
    
    print(f"Time step +{t+1}:")
    print(f"  MAE: {mae:.4f} µg/m³")
    print(f"  RMSE: {rmse:.4f} µg/m³")
    print(f"  R²: {r2:.4f}")
    print()

## 10. Visualize Predictions

In [ ]:
# Plot predictions vs actual for several examples
n_examples = 5
fig, axes = plt.subplots(n_examples, 1, figsize=(15, 3*n_examples))

for i in range(n_examples):
    idx = i * (len(y_test_denorm) // n_examples)
    
    time_steps = np.arange(1, n_steps_out + 1)
    
    axes[i].plot(time_steps, y_test_denorm[idx], 'o-', label='Actual', markersize=8, linewidth=2)
    axes[i].plot(time_steps, y_pred_denorm[idx], 's--', label='Predicted', markersize=8, linewidth=2)
    
    axes[i].set_xlabel('Hours Ahead')
    axes[i].set_ylabel('PM2.5 (µg/m³)')
    axes[i].set_title(f'Example {i+1}: 5-Hour Prediction')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)
    axes[i].set_xticks(time_steps)

plt.tight_layout()
plt.show()

In [ ]:
# Create a time series plot showing predictions over a longer period
# Show first 200 predictions
n_show = min(200, len(y_test_denorm))

fig, axes = plt.subplots(2, 1, figsize=(20, 10))

# Plot for 1-hour ahead predictions
axes[0].plot(y_test_denorm[:n_show, 0], label='Actual', alpha=0.7, linewidth=2)
axes[0].plot(y_pred_denorm[:n_show, 0], label='Predicted', alpha=0.7, linewidth=2)
axes[0].set_xlabel('Sample')
axes[0].set_ylabel('PM2.5 (µg/m³)')
axes[0].set_title('1-Hour Ahead Predictions')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot for 5-hour ahead predictions
axes[1].plot(y_test_denorm[:n_show, 4], label='Actual', alpha=0.7, linewidth=2)
axes[1].plot(y_pred_denorm[:n_show, 4], label='Predicted', alpha=0.7, linewidth=2)
axes[1].set_xlabel('Sample')
axes[1].set_ylabel('PM2.5 (µg/m³)')
axes[1].set_title('5-Hour Ahead Predictions')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots to show prediction quality
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1-hour ahead
axes[0].scatter(y_test_denorm[:, 0], y_pred_denorm[:, 0], alpha=0.5, s=10)
axes[0].plot([y_test_denorm[:, 0].min(), y_test_denorm[:, 0].max()], 
             [y_test_denorm[:, 0].min(), y_test_denorm[:, 0].max()], 
             'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual PM2.5 (µg/m³)')
axes[0].set_ylabel('Predicted PM2.5 (µg/m³)')
axes[0].set_title('1-Hour Ahead Prediction')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 3-hour ahead
axes[1].scatter(y_test_denorm[:, 2], y_pred_denorm[:, 2], alpha=0.5, s=10)
axes[1].plot([y_test_denorm[:, 2].min(), y_test_denorm[:, 2].max()], 
             [y_test_denorm[:, 2].min(), y_test_denorm[:, 2].max()], 
             'r--', linewidth=2, label='Perfect Prediction')
axes[1].set_xlabel('Actual PM2.5 (µg/m³)')
axes[1].set_ylabel('Predicted PM2.5 (µg/m³)')
axes[1].set_title('3-Hour Ahead Prediction')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 5-hour ahead
axes[2].scatter(y_test_denorm[:, 4], y_pred_denorm[:, 4], alpha=0.5, s=10)
axes[2].plot([y_test_denorm[:, 4].min(), y_test_denorm[:, 4].max()], 
             [y_test_denorm[:, 4].min(), y_test_denorm[:, 4].max()], 
             'r--', linewidth=2, label='Perfect Prediction')
axes[2].set_xlabel('Actual PM2.5 (µg/m³)')
axes[2].set_ylabel('Predicted PM2.5 (µg/m³)')
axes[2].set_title('5-Hour Ahead Prediction')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Summary and Conclusions

In [ ]:
print("="*60)
print("LSTM MODEL SUMMARY")
print("="*60)
print(f"\nModel Configuration:")
print(f"  Input: {n_steps_in} time steps ({n_features} features each)")
print(f"  Output: {n_steps_out} time steps (PM2.5 predictions)")
print(f"\nData Statistics:")
print(f"  Training samples: {len(X_train):,}")
print(f"  Validation samples: {len(X_val):,}")
print(f"  Test samples: {len(X_test):,}")
print(f"\nModel Performance (on test set):")
print(f"  Test Loss (MSE): {test_loss:.6f}")
print(f"  Test MAE: {test_mae:.6f}")

# Calculate overall statistics
overall_mae = mean_absolute_error(y_test_denorm.flatten(), y_pred_denorm.flatten())
overall_rmse = np.sqrt(mean_squared_error(y_test_denorm.flatten(), y_pred_denorm.flatten()))
overall_r2 = r2_score(y_test_denorm.flatten(), y_pred_denorm.flatten())

print(f"\nOverall Performance (all prediction horizons):")
print(f"  MAE: {overall_mae:.4f} µg/m³")
print(f"  RMSE: {overall_rmse:.4f} µg/m³")
print(f"  R²: {overall_r2:.4f}")

print(f"\nFeatures Used:")
for i, col in enumerate(feature_cols):
    print(f"  {i+1}. {col}")

print(f"\nTrust-based Data Filling:")
print(f"  Original observations: {(df_complete['fill_method'] == 'original').sum():,}")
print(f"  Forward filled: {(df_complete['fill_method'] == 'forward').sum():,}")
print(f"  Backward filled: {(df_complete['fill_method'] == 'backward').sum():,}")
print(f"  Average trust value: {df_complete['trust'].mean():.4f}")

print("\n" + "="*60)
print("Model training and evaluation complete!")
print("="*60)